In [19]:
from pathlib import Path

DATASET_DIR = '../dataset'

ROOT_DIR = Path().resolve().__str__()
DATASET_PATH = ROOT_DIR + DATASET_DIR
DATASET_FILENAME = 'Anvisa_Alertas_2026-01-01_2026-06-19.csv'
DATASET_FILEPATH = Path(DATASET_DIR + '/' + DATASET_FILENAME)
Path.mkdir(Path(DATASET_PATH), parents=True, exist_ok=True)

In [54]:
from requests import Response
from bs4 import BeautifulSoup
import cloudscraper as cs
import re

url = "https://antigo.anvisa.gov.br/alertas/-/buscar?p_p_id=anvisabuscaavancada_WAR_anvisabuscaavancadaportlet&keywords=&dataInicial=01/01/2016&dataFinal=19/06/2026&categoryIds=34506&categoryIds=5396434&categoryIds=5396432&categoryIds=34509&categoryIds=34510&categoryIds=34511"
page = "&pagina="
url_paginated = url+page

def scrap(url: str):
    scrapper = cs.create_scraper()
    res = scrapper.get(url)
    return res, BeautifulSoup(res.text, "html.parser")

res, soup = scrap(url)

alertas = soup.find_all(name='p', attrs={'class': 'titulo'})

# URLs
links = [alerta.find('a', href=True)['href'] for alerta in alertas]

# Titulo
titulos = [alerta.text.strip() for alerta in alertas]

# Data
def data(soup: BeautifulSoup):
    datas = soup.find_all('p', attrs={'class': 'data'})
    datas = [data.text.strip() for data in datas]
    datas = filter(lambda d: re.search(r"[a-z]", d) is None, datas)
    datas = list(datas)
    return datas

datas = data(soup)

# Paginação
pagination = soup.find_all(name='li', attrs={'class': 'pagination-control'})

In [52]:
last_page = max([int(page.find('a')['data-pagination'] if page.find('a')['data-pagination'] != '' else '0') for page in pagination])
next_page = 1

In [55]:
while next_page < last_page:
    next_page = next_page + 1
    res, soup = scrap(url_paginated+str(next_page))

    if res.status_code != 200:
        error_msg = f"Response code: {res.status_code} | Reason: {res.reason} | URL: {url_paginated+str(next_page)}"
        print(f"Response code: {res.status_code} | Reason: {res.reason} | URL: {url_paginated+str(next_page)}")
        links.append(error_msg)
        titulos.append(error_msg)
        datas.append(error_msg)
    else:
        alertas = soup.find_all(name='p', attrs={'class': 'titulo'})

        _links = [alerta.find('a', href=True)['href'] for alerta in alertas]
        for _link in _links:
            links.append(_link)

        _titulos = [alerta.text.strip() for alerta in alertas]
        for _titulo in _titulos:
            titulos.append(_titulo)

        _datas  = data(soup)
        for _data in _datas:
            datas.append(_data)


Response code: None | URL: https://antigo.anvisa.gov.br/alertas/-/buscar?p_p_id=anvisabuscaavancada_WAR_anvisabuscaavancadaportlet&keywords=&dataInicial=01/01/2016&dataFinal=19/06/2026&categoryIds=34506&categoryIds=5396434&categoryIds=5396432&categoryIds=34509&categoryIds=34510&categoryIds=34511&pagina=14
Response code: None | URL: https://antigo.anvisa.gov.br/alertas/-/buscar?p_p_id=anvisabuscaavancada_WAR_anvisabuscaavancadaportlet&keywords=&dataInicial=01/01/2016&dataFinal=19/06/2026&categoryIds=34506&categoryIds=5396434&categoryIds=5396432&categoryIds=34509&categoryIds=34510&categoryIds=34511&pagina=15
Response code: None | URL: https://antigo.anvisa.gov.br/alertas/-/buscar?p_p_id=anvisabuscaavancada_WAR_anvisabuscaavancadaportlet&keywords=&dataInicial=01/01/2016&dataFinal=19/06/2026&categoryIds=34506&categoryIds=5396434&categoryIds=5396432&categoryIds=34509&categoryIds=34510&categoryIds=34511&pagina=16
Response code: None | URL: https://antigo.anvisa.gov.br/alertas/-/buscar?p_p_id

KeyboardInterrupt: 

In [49]:
datas

['16/06/2026',
 '16/06/2026',
 '16/06/2026',
 '16/06/2026',
 '16/06/2026',
 '15/06/2026',
 '15/06/2026',
 '15/06/2026',
 '15/06/2026',
 '12/06/2026',
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 No

In [13]:
import pandas as pd

df = pd.DataFrame(data={'titulo': titulos, 'data': datas, 'url': links})
df

,titulo,data,url
0,Alerta 5265 (Tecnovigilância) - Comunicado da ...,16/06/2026,http://antigo.anvisa.gov.br/informacoes-tecnic...
1,Alerta 5264 (Tecnovigilância) - Comunicado da ...,16/06/2026,http://antigo.anvisa.gov.br/informacoes-tecnic...
2,Alerta 5263 (Tecnovigilância) - Comunicado da ...,16/06/2026,http://antigo.anvisa.gov.br/informacoes-tecnic...
3,Alerta 5262 (Tecnovigilância) - Comunicado da ...,16/06/2026,http://antigo.anvisa.gov.br/informacoes-tecnic...
4,Alerta 5261 (Tecnovigilância) - Comunicado da ...,16/06/2026,http://antigo.anvisa.gov.br/informacoes-tecnic...
...,...,...,...
3514,Alerta 1789 Atualizado (Tecnovigilância) - GE ...,13/01/2016,http://antigo.anvisa.gov.br/informacoes-tecnic...
3515,Alerta 1788,12/01/2016,http://antigo.anvisa.gov.br/informacoes-tecnic...
3516,Alerta 1787,12/01/2016,http://antigo.anvisa.gov.br/informacoes-tecnic...
3517,Alerta 1780,06/01/2016,http://antigo.anvisa.gov.br/informacoes-tecnic...


In [21]:
df.to_csv(DATASET_FILEPATH, index=False, encoding='utf-8')


In [233]:
links = [alerta.find('a', href=True)['href'] for alerta in alertas]
datas = soup.find_all('p', attrs={'class': 'data'})

In [24]:
import cloudscraper as cs

scrapper = cs.create_scraper()

def retrieve_content(_url: str):
    res = scrapper.get(_url)

    if res.status_code != 200:
        print(f"Response code: {res.status_code} | URL: {_url}")
        return res.status_code

    html = res.text
    soup = BeautifulSoup(html, "html.parser")
    content = soup.find('div', attrs={'class':'journal-content-article'})
    return content.text

c_content = [retrieve_content(url) for url in df['url']]

In [26]:
df = pd.DataFrame(data={'titulo': titulos, 'data': datas, 'url': links, 'content': c_content})
df

,titulo,data,url,content
0,Alerta 5265 (Tecnovigilância) - Comunicado da ...,16/06/2026,http://antigo.anvisa.gov.br/informacoes-tecnic...,Área: GGMON Número: 5265 Ano: 2026 ...
1,Alerta 5264 (Tecnovigilância) - Comunicado da ...,16/06/2026,http://antigo.anvisa.gov.br/informacoes-tecnic...,Área: GGMON Número: 5264 Ano: 2026 ...
2,Alerta 5263 (Tecnovigilância) - Comunicado da ...,16/06/2026,http://antigo.anvisa.gov.br/informacoes-tecnic...,Área: GGMON Número: 5263 Ano: 2026 ...
3,Alerta 5262 (Tecnovigilância) - Comunicado da ...,16/06/2026,http://antigo.anvisa.gov.br/informacoes-tecnic...,Área: GGMON Número: 5262 Ano: 2026 ...
4,Alerta 5261 (Tecnovigilância) - Comunicado da ...,16/06/2026,http://antigo.anvisa.gov.br/informacoes-tecnic...,Área: GGMON Número: 5261 Ano: 2026 ...
...,...,...,...,...
3514,Alerta 1789 Atualizado (Tecnovigilância) - GE ...,13/01/2016,http://antigo.anvisa.gov.br/informacoes-tecnic...,Área: GGMON Número: 1789 Ano: 2016 ...
3515,Alerta 1788,12/01/2016,http://antigo.anvisa.gov.br/informacoes-tecnic...,Área: GGMON Número: 1788 Ano: 2016 ...
3516,Alerta 1787,12/01/2016,http://antigo.anvisa.gov.br/informacoes-tecnic...,Área: GGMON Número: 1787 Ano: 2016 ...
3517,Alerta 1780,06/01/2016,http://antigo.anvisa.gov.br/informacoes-tecnic...,Área: GGMON Número: 1780 Ano: 2016 ...


In [79]:
duplicateRows_content = df[df.duplicated(['content'])]

In [106]:
duplicateRows_titulo = df[df.duplicated(['titulo'])]
duplicateRows_titulo

,titulo,data,url,content
1070,Alerta 4227 (Tecnovigilância) - Comunicado da ...,14/08/2023,http://antigo.anvisa.gov.br/informacoes-tecnic...,Área: GGMON Número: 4227 Ano: 2023 ...
1800,ALERTA: Anvisa alerta os profissionais de saúd...,04/05/2021,http://antigo.anvisa.gov.br/informacoes-tecnic...,Área: GGMON Número: 142021 Ano: 202...
3368,Siemens - Sistema de Ultrassom Diagnóstico ACU...,04/08/2016,http://antigo.anvisa.gov.br/informacoes-tecnic...,Área: GGMON Número: 1964 Ano: 2016 ...


In [101]:
dupes = []
for titulo in duplicateRows_titulo['titulo']:
    dupes.append(df.query('titulo == "' + titulo + '"')[['titulo', 'url', 'content']])

In [102]:
dupes[0]

,titulo,url,content
1069,Alerta 4227 (Tecnovigilância) - Comunicado da ...,http://antigo.anvisa.gov.br/informacoes-tecnic...,Área: GGMON Número: 4227 Ano: 2023 ...
1070,Alerta 4227 (Tecnovigilância) - Comunicado da ...,http://antigo.anvisa.gov.br/informacoes-tecnic...,Área: GGMON Número: 4227 Ano: 2023 ...


In [103]:
dupes[1]

,titulo,url,content
1744,ALERTA: Anvisa alerta os profissionais de saúd...,http://antigo.anvisa.gov.br/informacoes-tecnic...,Área: GGMON Número: 62021 Ano: 2021...
1800,ALERTA: Anvisa alerta os profissionais de saúd...,http://antigo.anvisa.gov.br/informacoes-tecnic...,Área: GGMON Número: 142021 Ano: 202...


In [104]:
dupes[2]

,titulo,url,content
3367,Siemens - Sistema de Ultrassom Diagnóstico ACU...,http://antigo.anvisa.gov.br/informacoes-tecnic...,Área: GGMON Número: 1965 Ano: 2016 ...
3368,Siemens - Sistema de Ultrassom Diagnóstico ACU...,http://antigo.anvisa.gov.br/informacoes-tecnic...,Área: GGMON Número: 1964 Ano: 2016 ...


In [27]:
df.to_csv(DATASET_FILEPATH, index=False, encoding='utf-8')